# PEID known-dynamics mediation example

This notebook builds a classic mediation system with known dynamics and reads it with PEID-style maximum-entropy interventions. The target mechanism is a single-source mediation chain `x -> m -> z`, not a synergistic mechanism.

The discrete-time continuous dynamics are

```text
x_{t+1} = a_x x_t + eta_x
m_{t+1} = a_m m_t + b_x tanh(x_t) + eta_m
z_{t+1} = a_z z_t + b_m tanh(m_t) + d_x tanh(x_t) + eta_z
```

Default `d_x = 0`, so the two-step influence from `x_t` to `z_{t+2}` must pass through `m_{t+1}`. The PEID analysis uses independent maximum-entropy interventions over a bounded source box, then estimates effective information after discretizing source and target states.

Literature grounding: the PEID definition follows the local Zotero preprint *Partial Effective Information Decomposition for Synergistic Causality* (Zotero item key `MYATYWAJ`), especially the intervention-based definition of effective information and the source-side maximum-entropy intervention semantics.


In [1]:
from __future__ import annotations

from dataclasses import dataclass, replace
from pathlib import Path
from typing import Iterable

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 8,
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.linewidth": 0.8,
    "legend.frameon": False,
})

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "exp":
    REPO_ROOT = REPO_ROOT.parent
FIGURE_DIR = REPO_ROOT / "fig" / "mediated_peid_known_dynamics"
FIGURE_PATH = FIGURE_DIR / "mediated_peid_known_dynamics.png"
FIGURE_MARKDOWN_PATH = "fig/mediated_peid_known_dynamics/mediated_peid_known_dynamics.png"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


## Known transition mechanism

The distractor `u` is included as a negative control. It is sampled under the same maximum-entropy intervention as the causal variables but does not enter the transition equations.


In [2]:
@dataclass(frozen=True)
class MediatedDynamicsConfig:
    a_x: float = 0.35
    a_m: float = 0.30
    a_z: float = 0.25
    b_x: float = 1.20
    b_m: float = 1.15
    d_x: float = 0.0
    intervention_low: float = -2.0
    intervention_high: float = 2.0
    n_intervention: int = 12000
    bins: int = 8
    seed: int = 20260602


def sample_max_entropy_sources(config: MediatedDynamicsConfig) -> pd.DataFrame:
    rng = np.random.default_rng(config.seed)
    values = rng.uniform(
        config.intervention_low,
        config.intervention_high,
        size=(config.n_intervention, 4),
    )
    return pd.DataFrame(values, columns=["x", "m", "z", "u"])


def one_step_transition(states: pd.DataFrame, config: MediatedDynamicsConfig, *, block_mediator: bool = False) -> pd.DataFrame:
    x = states["x"].to_numpy(dtype=float)
    m = states["m"].to_numpy(dtype=float)
    z = states["z"].to_numpy(dtype=float)
    u = states["u"].to_numpy(dtype=float)

    next_x = config.a_x * x
    next_m = config.a_m * m + config.b_x * np.tanh(x)
    mediator_term = 0.0 if block_mediator else config.b_m * np.tanh(m)
    next_z = config.a_z * z + mediator_term + config.d_x * np.tanh(x)
    next_u = 0.10 * u
    return pd.DataFrame({"x": next_x, "m": next_m, "z": next_z, "u": next_u})


def two_step_transition(states: pd.DataFrame, config: MediatedDynamicsConfig, *, block_second_step_mediator: bool = False) -> pd.DataFrame:
    first = one_step_transition(states, config)
    return one_step_transition(first, config, block_mediator=block_second_step_mediator)


## PEID-style EI estimator

For a source set `A` and target `B`, effective information is computed as `I(A_t; B_{t+1})` under independent maximum-entropy interventions. For the second-order negative control, the notebook reports

```text
Syn({x,m}->z) = EI({x,m}->z) - EI(x->z) - EI(m->z)
```

on the one-step target `z_{t+1}`. Because default `d_x = 0`, `x_t` has no direct one-step path to `z_{t+1}`, so the additive classic mediation setup should not create a strong one-step synergy hyperedge.


In [3]:
def discretize(values: np.ndarray, bins: int) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    edges = np.quantile(values, np.linspace(0.0, 1.0, bins + 1))
    edges = np.unique(edges)
    if len(edges) <= 2:
        return np.zeros(len(values), dtype=int)
    return np.digitize(values, edges[1:-1], right=False).astype(int)


def encode_joint_columns(frame: pd.DataFrame, columns: Iterable[str], bins: int) -> np.ndarray:
    codes = [discretize(frame[column].to_numpy(dtype=float), bins) for column in columns]
    if len(codes) == 1:
        return codes[0]
    return np.ravel_multi_index(codes, dims=tuple([bins] * len(codes)))


def mutual_information_from_codes(source_code: np.ndarray, target_code: np.ndarray) -> float:
    source_code = np.asarray(source_code, dtype=int)
    target_code = np.asarray(target_code, dtype=int)
    joint = pd.crosstab(source_code, target_code).to_numpy(dtype=float)
    joint /= joint.sum()
    px = joint.sum(axis=1, keepdims=True)
    py = joint.sum(axis=0, keepdims=True)
    expected = px @ py
    mask = joint > 0.0
    return float(np.sum(joint[mask] * np.log2(joint[mask] / expected[mask])))


def effective_information(source_states: pd.DataFrame, target_states: pd.DataFrame, sources: tuple[str, ...], target: str, bins: int) -> float:
    source_code = encode_joint_columns(source_states, sources, bins)
    target_code = discretize(target_states[target].to_numpy(dtype=float), bins)
    return mutual_information_from_codes(source_code, target_code)


def peid_readout(config: MediatedDynamicsConfig) -> dict[str, float]:
    states = sample_max_entropy_sources(config)
    one_step = one_step_transition(states, config)
    two_step_full = two_step_transition(states, config)
    two_step_blocked = two_step_transition(states, config, block_second_step_mediator=True)
    direct_only_config = replace(config, b_m=0.0)
    two_step_direct_only = two_step_transition(states, direct_only_config)

    one_step_edges = {
        "x_to_m": effective_information(states, one_step, ("x",), "m", config.bins),
        "m_to_z": effective_information(states, one_step, ("m",), "z", config.bins),
        "x_to_z": effective_information(states, one_step, ("x",), "z", config.bins),
        "u_to_z": effective_information(states, one_step, ("u",), "z", config.bins),
        "xm_to_z": effective_information(states, one_step, ("x", "m"), "z", config.bins),
    }
    one_step_edges["xm_synergy_to_z"] = one_step_edges["xm_to_z"] - one_step_edges["x_to_z"] - one_step_edges["m_to_z"]

    two_step_edges = {
        "x_to_z_full": effective_information(states, two_step_full, ("x",), "z", config.bins),
        "x_to_z_mediator_blocked": effective_information(states, two_step_blocked, ("x",), "z", config.bins),
        "x_to_z_direct_only": effective_information(states, two_step_direct_only, ("x",), "z", config.bins),
        "u_to_z_full": effective_information(states, two_step_full, ("u",), "z", config.bins),
    }
    return {**one_step_edges, **two_step_edges}


BASE_CONFIG = MediatedDynamicsConfig()
readout = peid_readout(BASE_CONFIG)
readout


{'x_to_m': 1.2025041000175432,
 'm_to_z': 1.2714366421864416,
 'x_to_z': 0.002176275828833951,
 'u_to_z': 0.002909921330237448,
 'xm_to_z': 1.2800428597250633,
 'xm_synergy_to_z': 0.006429941709787723,
 'x_to_z_full': 0.8493332959910866,
 'x_to_z_mediator_blocked': 0.002176275828833951,
 'x_to_z_direct_only': 0.0028933644398857995,
 'u_to_z_full': 0.0031336011899479686}

## Direct-effect sensitivity

The sweep increases `d_x`, the direct `x_t -> z_{t+1}` path. If the blocking control is working, the mediator-blocked two-step EI stays low when `d_x = 0` and grows as direct influence is added.


In [4]:
def direct_effect_sweep(direct_values: Iterable[float], base_config: MediatedDynamicsConfig) -> pd.DataFrame:
    rows: list[dict[str, float]] = []
    for value in direct_values:
        config = replace(base_config, d_x=float(value), seed=base_config.seed + int(round(float(value) * 1000)))
        result = peid_readout(config)
        rows.append({"d_x": float(value), **result})
    return pd.DataFrame(rows)


sweep = direct_effect_sweep(np.linspace(0.0, 0.6, 7), BASE_CONFIG)
sweep[["d_x", "x_to_z_full", "x_to_z_mediator_blocked", "x_to_z_direct_only", "u_to_z_full"]]


,d_x,x_to_z_full,x_to_z_mediator_blocked,x_to_z_direct_only,u_to_z_full
0,0.0,0.849333,0.002176,0.002893,0.003134
1,0.1,0.883161,0.091024,0.380498,0.002811
2,0.2,0.931502,0.273470,0.834390,0.002176
3,0.3,0.984776,0.428036,1.153079,0.002891
4,0.4,1.027940,0.567247,1.400121,0.002900
5,0.5,1.060213,0.711298,1.568257,0.003792
6,0.6,1.112915,0.856399,1.689390,0.003315


## Numeric acceptance checks

`MEDIATED_PEID_NUMERIC_CHECKS` records the notebook-level acceptance tests: the two one-step path edges are positive, two-step `x -> z` drops when the mediator is blocked, the distractor stays near zero, and the additive classic mediation setup does not produce a strong one-step `{x,m}->z` synergy hyperedge.


In [5]:
MEDIATED_PEID_NUMERIC_CHECKS = {
    "x_to_m_positive": readout["x_to_m"] > 0.25,
    "m_to_z_positive": readout["m_to_z"] > 0.25,
    "mediator_blocked_drop": readout["x_to_z_mediator_blocked"] < 0.35 * readout["x_to_z_full"],
    "distractor_ei_low": readout["u_to_z_full"] < 0.03 and readout["u_to_z"] < 0.03,
    "additive_synergy_low": abs(readout["xm_synergy_to_z"]) < 0.03,
}
assert all(MEDIATED_PEID_NUMERIC_CHECKS.values()), {"checks": MEDIATED_PEID_NUMERIC_CHECKS, "readout": readout}

pd.DataFrame(
    [
        {"quantity": "EI x_t -> m_{t+1}", "value_bits": readout["x_to_m"]},
        {"quantity": "EI m_t -> z_{t+1}", "value_bits": readout["m_to_z"]},
        {"quantity": "EI x_t -> z_{t+2} full", "value_bits": readout["x_to_z_full"]},
        {"quantity": "EI x_t -> z_{t+2} mediator_blocked", "value_bits": readout["x_to_z_mediator_blocked"]},
        {"quantity": "EI u_t -> z_{t+2} distractor", "value_bits": readout["u_to_z_full"]},
        {"quantity": "Syn {x,m}->z one-step", "value_bits": readout["xm_synergy_to_z"]},
    ]
)


,quantity,value_bits
0,EI x_t -> m_{t+1},1.202504
1,EI m_t -> z_{t+1},1.271437
2,EI x_t -> z_{t+2} full,0.849333
3,EI x_t -> z_{t+2} mediator_blocked,0.002176
4,EI u_t -> z_{t+2} distractor,0.003134
5,"Syn {x,m}->z one-step",0.006430


## Figure

The figure is saved to `fig/mediated_peid_known_dynamics/mediated_peid_known_dynamics.png`.


In [6]:
def plot_mediated_peid_summary(readout: dict[str, float], sweep: pd.DataFrame, output_path: Path) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(9.2, 5.8), constrained_layout=True)
    ax_graph, ax_one, ax_two, ax_sweep = axes.ravel()

    ax_graph.axis("off")
    node_pos = {"x": (0.12, 0.55), "m": (0.50, 0.55), "z": (0.88, 0.55), "u": (0.50, 0.18)}
    for name, (x_pos, y_pos) in node_pos.items():
        face = "#f2c96d" if name in {"x", "m", "z"} else "#d9d9d9"
        ax_graph.scatter([x_pos], [y_pos], s=900, color=face, edgecolor="#333333", linewidth=1.0, zorder=3)
        ax_graph.text(x_pos, y_pos, name, ha="center", va="center", fontsize=10, fontweight="bold")
    arrow_style = dict(arrowstyle="->", lw=1.4, color="#333333", shrinkA=16, shrinkB=16)
    ax_graph.annotate("", xy=node_pos["m"], xytext=node_pos["x"], arrowprops=arrow_style)
    ax_graph.annotate("", xy=node_pos["z"], xytext=node_pos["m"], arrowprops=arrow_style)
    ax_graph.annotate("", xy=node_pos["z"], xytext=node_pos["x"], arrowprops={**arrow_style, "linestyle": ":", "color": "#9a5a5a", "connectionstyle": "arc3,rad=-0.32"})
    ax_graph.text(0.50, 0.78, "Known mediation dynamics", ha="center", va="center", fontsize=9, fontweight="bold")
    ax_graph.text(0.50, 0.04, "dotted edge: optional direct path d_x", ha="center", va="center", fontsize=7, color="#666666")
    ax_graph.set_xlim(0, 1)
    ax_graph.set_ylim(0, 1)

    one_step_labels = ["x -> m", "m -> z", "x -> z", "u -> z", "Syn {x,m}->z"]
    one_step_values = [readout["x_to_m"], readout["m_to_z"], readout["x_to_z"], readout["u_to_z"], readout["xm_synergy_to_z"]]
    one_colors = ["#6f9ecf", "#6aa36f", "#c9c9c9", "#c9c9c9", "#d99a48"]
    ax_one.barh(one_step_labels, one_step_values, color=one_colors)
    ax_one.axvline(0, color="#333333", lw=0.8)
    ax_one.set_xlabel("EI / synergy (bits)")
    ax_one.set_title("One-step PEID readout", fontsize=9, fontweight="bold")

    two_step_labels = ["full", "mediator_blocked", "direct_only", "distractor"]
    two_step_values = [
        readout["x_to_z_full"],
        readout["x_to_z_mediator_blocked"],
        readout["x_to_z_direct_only"],
        readout["u_to_z_full"],
    ]
    ax_two.bar(two_step_labels, two_step_values, color=["#6f9ecf", "#d9a0a0", "#c9c9c9", "#c9c9c9"])
    ax_two.set_ylabel("EI x_t -> z_{t+2} (bits)")
    ax_two.set_title("Two-step blocking control", fontsize=9, fontweight="bold")
    ax_two.tick_params(axis="x", rotation=20)

    ax_sweep.plot(sweep["d_x"], sweep["x_to_z_full"], marker="o", label="full", color="#6f9ecf")
    ax_sweep.plot(sweep["d_x"], sweep["x_to_z_mediator_blocked"], marker="s", label="mediator blocked", color="#d9a0a0")
    ax_sweep.plot(sweep["d_x"], sweep["x_to_z_direct_only"], marker="^", label="direct only", color="#6aa36f")
    ax_sweep.plot(sweep["d_x"], sweep["u_to_z_full"], marker="x", label="distractor", color="#777777")
    ax_sweep.set_xlabel("direct path strength d_x")
    ax_sweep.set_ylabel("EI x_t -> z_{t+2} (bits)")
    ax_sweep.set_title("Direct-effect sensitivity", fontsize=9, fontweight="bold")
    ax_sweep.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)

    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


plot_mediated_peid_summary(readout, sweep, FIGURE_PATH)
FIGURE_MARKDOWN_PATH


'fig/mediated_peid_known_dynamics/mediated_peid_known_dynamics.png'

## Interpretation

The one-step PEID edges identify the local mechanism: `x_t -> m_{t+1}` and `m_t -> z_{t+1}` are strong, while the distractor is near zero. The two-step PEID readout identifies the total mediated influence `x_t -> z_{t+2}`. When the mediator contribution is blocked in the second step, that influence drops sharply at `d_x = 0`, which separates the mediated path from a direct path. As `d_x` increases, the blocked and direct-only controls rise, showing that the same PEID intervention readout can distinguish mediation from direct influence.

![PEID known-dynamics mediation summary](../fig/mediated_peid_known_dynamics/mediated_peid_known_dynamics.png)
